# Kepler's First Law

## The law

A planet's path around the Sun is an **ellipse**. The Sun does not sit at the center of that ellipse. It sits at one **focus**.

An ellipse is a stretched circle. It has a center, and two foci at equal distances on either side of that center. The closer the foci are to the center, the rounder the ellipse looks. If both foci lie on the center, the path is a circle. A circle is just the special ellipse where the Sun happens to be in the middle.

## Closest and farthest

- **Periapsis** is the closest point to the Sun. The Sun's focus is nearer this end.
- **Apoapsis** is the farthest point from the Sun.
- **a**, the semi-major axis, is half the long axis of the ellipse. It is half the distance from periapsis to apoapsis.
- **Eccentricity** measures the stretch. It is 0 for a circle. Earth's orbit is about 0.017, so it is almost a circle. The orbit drawn here is more stretched, so you can see that the Sun is offset.

## A check

Every point on an ellipse has the same total distance to the two foci: distance to the Sun plus distance to the empty focus equals **2a**, the full long axis. The simulation should agree with that.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np

from src.physics import AU, M_SUN, circular_orbital_speed, kepler_period
from src.simulation import integrate_two_body
from src.visualization import apply_dark_style, focus_geometry, plot_kepler_first_law

%matplotlib inline
apply_dark_style()


## One stretched orbit

The planet starts at 1 AU, moving sideways at 65% of the speed a circular orbit would need at that distance. It is not going fast enough to stay on a circle, so it falls closer to the Sun and then swings back out.

The Sun stays fixed at the origin. That origin is one focus. The center of the ellipse is somewhere else.


In [ ]:
r0 = AU
speed = 0.65 * circular_orbital_speed(r0, M_SUN)
dt = 3600.0
n_steps = int(1.4 * kepler_period(0.7 * r0, M_SUN) / dt)
history = integrate_two_body(
    np.array([r0, 0.0, 0.0]),
    np.array([0.0, speed, 0.0]),
    M_SUN,
    dt,
    n_steps,
)

geometry = focus_geometry(history)
a_au = geometry["semi_major_axis"] / AU
print(f"semi-major axis a = {a_au:.3f} AU")
print(f"eccentricity      = {geometry['eccentricity']:.3f}")
print(f"center is {np.linalg.norm(geometry['center']) / AU:.3f} AU from the Sun")

positions = history.positions[::80, :2]
sums = np.linalg.norm(positions - geometry["sun"], axis=1)
sums = sums + np.linalg.norm(positions - geometry["empty_focus"], axis=1)
print(f"distance to both foci / (2a) ranges from {sums.min() / (2 * geometry['semi_major_axis']):.4f} to {sums.max() / (2 * geometry['semi_major_axis']):.4f}")

fig = plot_kepler_first_law(history)
plt.show()


## What the figure shows

The gold dot is the Sun. The open dot is the empty focus. The plus sign is the center of the ellipse.

- **a** is half the long axis, from the center out to apoapsis.
- **b** is half the short axis.
- **c** is the distance from the center to the Sun. On a circle, **c** is zero and the Sun is the center.

The Sun is closer to periapsis than to apoapsis. That is the first law: the Sun is a focus, not the center.

## The planet moving

The same ellipse, with the planet traveling around it. It moves faster near the Sun and slower at the far end. The Sun stays at the focus.

<div id="ellipse-stage"></div>

This model assumes one planet, a fixed Sun, and Newtonian gravity. Real planetary orbits are ellipses too, but Earth's is so close to a circle that the offset of the Sun is small.
